VarExtractor

In [33]:
import ast


class CryptoVarExtractor(ast.NodeTransformer):

    def __init__(self):
        self.next_var_id = 0

        self.scope_stack = []

    def _new_scope(self):
        scope = {"mapping": {}, "assigns": []}
        self.scope_stack.append(scope)

    def _end_scope(self):
        return self.scope_stack.pop()

    def _make_var_name(self):
        name = f"cond_{self.next_var_id}"
        self.next_var_id += 1
        return name


    def _replace_condition(self, node):

        if isinstance(node, ast.Name) and node.id.startswith("cond_"):
            return node

        scope = self.scope_stack[-1]

        if id(node) not in scope["mapping"]:
            var_name = self._make_var_name()
            scope["mapping"][id(node)] = var_name

            assign = ast.Assign(
                targets=[ast.Name(id=var_name, ctx=ast.Store())],
                value=node,
            )
            ast.fix_missing_locations(assign)
            scope["assigns"].append(assign)

        return ast.Name(
            id=scope["mapping"][id(node)],
            ctx=ast.Load(),
        )

    def visit_Module(self, node):
        self._new_scope()

        self.generic_visit(node)

        scope = self._end_scope()

        node.body = scope["assigns"] + node.body
        return node

    def visit_FunctionDef(self, node):
        self._new_scope()

        self.generic_visit(node)

        scope = self._end_scope()
        node.body = scope["assigns"] + node.body

        return node

    def visit_If(self, node):
        self.generic_visit(node)

        if not isinstance(node.test, ast.Name):
            node.test = self._replace_condition(node.test)

        return node

    def visit_BoolOp(self, node):
        self.generic_visit(node)
        return self._replace_condition(node)

    def visit_Compare(self, node):
        self.generic_visit(node)
        return self._replace_condition(node)

    def get_refactored_code(self, source_code):
        try:
            tree = ast.parse(source_code)
            self.next_var_id = 0

            new_tree = self.visit(tree)
            ast.fix_missing_locations(new_tree)

            return ast.unparse(new_tree)

        except Exception as e:
            raise ValueError(f"Parse transform failed: {e}")


if __name__ == "__main__":
    src = """ 
a = 1
b = 4 
c = 3
d = 2

def fun():
    if a > b and (a != c) or (d != b):
        print("condition check 2")

if (a>b) or ((a !=b) or (c < d)) or ((c > a) and (b > d)):
    print("condition check")
else:
    pass
"""

    extractor = CryptoVarExtractor()
    print(extractor.get_refactored_code(src))
    
    src = """ 
a = 1
b = 4 
c = 3
d = 2

def fun():
    if a > b and (a != c) or (d != b):
        print("condition check 2")

if (a>b) or ((a !=b) or (c < d)) or ((c > a) and (b > d)):
    print("condition check")
else:
    pass
"""
    print("\n**********************************\n")
    extractor = CryptoVarExtractor()
    print(extractor.get_refactored_code(src))
    src = """ 
def manual_shared_key(key, mode, ephermal_key=None):
    if mode=="encrypt":
        ephermal_pub, shared_secret = Kyber().encaps(key)
        return (SHA384.new(shared_secret).digest()[:32], ephermal_pub)
    elif mode=="decrypt":
        shared_secret = Kyber().decaps(key, ephermal_key)
        return (SHA384.new(shared_secret).digest()[:32], None)
"""
    print("\n**********************************\n")
    extractor = CryptoVarExtractor()
    print(extractor.get_refactored_code(src))

cond_5 = a > b
cond_6 = a != b
cond_7 = c < d
cond_8 = cond_6 or cond_7
cond_9 = c > a
cond_10 = b > d
cond_11 = cond_9 and cond_10
cond_12 = cond_5 or cond_8 or cond_11
a = 1
b = 4
c = 3
d = 2

def fun():
    cond_0 = a > b
    cond_1 = a != c
    cond_2 = cond_0 and cond_1
    cond_3 = d != b
    cond_4 = cond_2 or cond_3
    if cond_4:
        print('condition check 2')
if cond_12:
    print('condition check')
else:
    pass

**********************************

cond_5 = a > b
cond_6 = a != b
cond_7 = c < d
cond_8 = cond_6 or cond_7
cond_9 = c > a
cond_10 = b > d
cond_11 = cond_9 and cond_10
cond_12 = cond_5 or cond_8 or cond_11
a = 1
b = 4
c = 3
d = 2

def fun():
    cond_0 = a > b
    cond_1 = a != c
    cond_2 = cond_0 and cond_1
    cond_3 = d != b
    cond_4 = cond_2 or cond_3
    if cond_4:
        print('condition check 2')
if cond_12:
    print('condition check')
else:
    pass

**********************************

def manual_shared_key(key, mode, ephermal_key=None):
    cond_

elseif and elif

In [5]:
import libcst as cst


class ElseTransformer(cst.CSTTransformer):

    def leave_If(self, original: cst.If, updated: cst.If):

        if isinstance(updated.orelse, cst.If):
            return self._elif_to_else_if(updated)

        if self._is_else_if(updated):
            return self._else_if_to_elif(updated)

        return updated

    def _is_else_if(self, node: cst.If):
        orelse = node.orelse

        return (
            isinstance(orelse, cst.Else)
            and len(orelse.body.body) == 1
            and isinstance(orelse.body.body[0], cst.If)
        )

    def _elif_to_else_if(self, node: cst.If) -> cst.If:

        return node.with_changes(
            orelse=cst.Else(
                body=cst.IndentedBlock(
                    body=[node.orelse]
                )
            )
        )

    def _else_if_to_elif(self, node: cst.If) -> cst.If:

        inner_if = node.orelse.body.body[0].with_changes(
            leading_lines=[]
        )

        return node.with_changes(
            orelse=inner_if
        )

    def get_refactored_code(self, source: str) -> str:
        module = cst.parse_module(source)
        return module.visit(self).code




if __name__ == "__main__":
    src = """
if a > b:
    print("A")
elif a == b:
    print("Equal")
else:
    if a < b:
        print("B")


if x > y:
    print("X")
else:
    if x == y:
        print("EQ")
    elif x < y:
        print("Y")

if x < y:
    print("x < y")
elif x != y:
    print("x != y")
else:
    print("x > y")
cond0 = x < y
cond1 = x == y

def main():
    if cond0:
        print("x < y")
    elif cond1:
        print("x != y")


if x > y:
    print("X")
else:
    if x == y:
        print("EQ")
    else:
        if x < y:
            print("Y")

"""

    transformer = ElseTransformer()
    print(transformer.get_refactored_code(src))
    src = """ 
def manual_shared_key(key, mode, ephermal_key=None):
    if mode=="encrypt":
        ephermal_pub, shared_secret = Kyber().encaps(key)
        return (SHA384.new(shared_secret).digest()[:32], ephermal_pub)
    elif mode=="decrypt":
        shared_secret = Kyber().decaps(key, ephermal_key)
        return (SHA384.new(shared_secret).digest()[:32], None)
"""
    print("\n**********************************\n")
    extractor = ElseTransformer()
    print(extractor.get_refactored_code(src))


if a > b:
    print("A")
else:
    if a == b:
        print("Equal")
    elif a < b:
        print("B")


if x > y:
    print("X")
elif x == y:
    print("EQ")
else:
    if x < y:
        print("Y")

if x < y:
    print("x < y")
else:
    if x != y:
        print("x != y")
    else:
        print("x > y")
cond0 = x < y
cond1 = x == y

def main():
    if cond0:
        print("x < y")
    else:
        if cond1:
            print("x != y")


if x > y:
    print("X")
elif x == y:
    print("EQ")
elif x < y:
    print("Y")



**********************************

 
def manual_shared_key(key, mode, ephermal_key=None):
    if mode=="encrypt":
        ephermal_pub, shared_secret = Kyber().encaps(key)
        return (SHA384.new(shared_secret).digest()[:32], ephermal_pub)
    else:
        if mode=="decrypt":
            shared_secret = Kyber().decaps(key, ephermal_key)
            return (SHA384.new(shared_secret).digest()[:32], None)



Relation operators

In [10]:
import ast


class RelationRefactors(ast.NodeTransformer):

    def change_exp(self, node: ast.Compare):
        left = node.left
        op = node.ops[0]
        right = node.comparators[0]

        def swapped(new_op):
            new_node = ast.Compare(
                left=right,
                ops=[new_op],
                comparators=[left]
            )
            return ast.copy_location(new_node, node)

        if isinstance(op, ast.Gt):
            return swapped(ast.Lt())

        elif isinstance(op, ast.GtE):
            return swapped(ast.LtE())

        elif isinstance(op, ast.Lt):
            return swapped(ast.Gt())

        elif isinstance(op, ast.LtE):
            return swapped(ast.GtE())

        elif isinstance(op, ast.Eq):
            new_node = ast.UnaryOp(
                op=ast.Not(),
                operand=ast.Compare(
                    left=left,
                    ops=[ast.NotEq()],
                    comparators=[right]
                )
            )
            return ast.copy_location(new_node, node)

        elif isinstance(op, ast.NotEq):
            new_node = ast.UnaryOp(
                op=ast.Not(),
                operand=ast.Compare(
                    left=left,
                    ops=[ast.Eq()],
                    comparators=[right]
                )
            )
            return ast.copy_location(new_node, node)

        return node  

    def conditional_check(self, node):

        if isinstance(node, ast.Compare):
            return self.change_exp(node)

        elif isinstance(node, ast.BoolOp):
            return ast.BoolOp(
                op=node.op,
                values=[self.conditional_check(v) for v in node.values]
            )

        return node

    def visit_If(self, node):
        self.generic_visit(node)
        node.test = self.conditional_check(node.test)
        return node

    def visit_While(self, node):
        self.generic_visit(node)
        node.test = self.conditional_check(node.test)
        return node

    def get_refactored_code(self, code):
        try:
            tree = ast.parse(code)
            tree = self.visit(tree)
            ast.fix_missing_locations(tree)
            return ast.unparse(tree)
        except SyntaxError as e:
            raise ValueError(f"Syntax error in source code: {e}")


if __name__ == "__main__":

    src = """
def func(a, b, c):
    if a>b and b >= c or a==c:
        return a
    elif c < b:
        return b
    elif a == c:
        return c
    elif func2(): return None

while(a<b and var1!=var2):
    a += 1
"""

    trans = RelationRefactors()
    print(trans.get_refactored_code(src))

    print("\n**********************************\n")

    src = """ 
def manual_shared_key(key, mode, ephermal_key=None):
    if mode=="encrypt":
        ephermal_pub, shared_secret = Kyber().encaps(key)
        return (SHA384.new(shared_secret).digest()[:32], ephermal_pub)
    elif mode=="decrypt":
        shared_secret = Kyber().decaps(key, ephermal_key)
        return (SHA384.new(shared_secret).digest()[:32], None)
"""

    extractor = RelationRefactors()
    print(extractor.get_refactored_code(src))

def func(a, b, c):
    if b < a and c <= b or not a != c:
        return a
    elif b > c:
        return b
    elif not a != c:
        return c
    elif func2():
        return None
while b > a and (not var1 == var2):
    a += 1

**********************************

def manual_shared_key(key, mode, ephermal_key=None):
    if not mode != 'encrypt':
        (ephermal_pub, shared_secret) = Kyber().encaps(key)
        return (SHA384.new(shared_secret).digest()[:32], ephermal_pub)
    elif not mode != 'decrypt':
        shared_secret = Kyber().decaps(key, ephermal_key)
        return (SHA384.new(shared_secret).digest()[:32], None)


Assertions, #try again if assertions is already added before

In [6]:
import ast
import random
import re



class AddAssertions(ast.NodeTransformer):
    def visit_FunctionDef(self, node):
        args = node.args.args
        defaults = node.args.defaults
        offset = len(args) - len(defaults)

        params_to_assert = []

        for i, arg in enumerate(args):
            if arg.arg == 'self':
                continue

            default = None
            if i >= offset:
                default = defaults[i - offset]

            if not (isinstance(default, ast.Constant) and default.value is None):
                params_to_assert.append(arg.arg)

        if params_to_assert:
            conds = [
                ast.Compare(
                    left=ast.Name(id=p, ctx=ast.Load()),
                    ops=[ast.NotEq()],
                    comparators=[ast.Constant(None)]
                )
                for p in params_to_assert
            ]
            test = conds[0] if len(conds) == 1 else ast.BoolOp(op=ast.And(), values=conds)
            node.body.insert(0, ast.Assert(test=test))

        return self.generic_visit(node)

    def get_refactored_code(self, source_code):
        try:
            tree = ast.parse(source_code)
            tree = self.visit(tree)
            result = ast.unparse(tree)
        except SyntaxError as e:
            raise ValueError(f"Syntax error in source code: {e}")

        

        return result
if __name__ == "__main__":
    src = """
def process(a, b, c=None):
    return a + b

def add(a, b):
    return a + b

def load(path, encoding=None):
    print(path)

class A:
    def f(self, x, y=None):
        return x






"""
    trans = AddAssertions()
    print(trans.get_refactored_code(src))


def process(a, b, c=None):
    assert a != None and b != None
    return a + b

def add(a, b):
    assert a != None and b != None
    return a + b

def load(path, encoding=None):
    assert path != None
    print(path)

class A:

    def f(self, x, y=None):
        assert x != None
        return x


ID Generalization

In [13]:
import ast

class idCollector(ast.NodeVisitor):

    def __init__(self):
        self.ids = set()
        self.func_ids = set()
        self.class_ids = set()

    def visit_ClassDef(self, node):
        self.class_ids.add(node.name)
        self.generic_visit(node)

    def visit_FunctionDef(self, node):
        self.func_ids.add(node.name)

        for arg in node.args.args:
            if arg.arg != "self":
                self.ids.add(arg.arg)

        self.generic_visit(node)

    def visit_Assign(self, node):
        for target in node.targets:

            if isinstance(target, ast.Name):
                self.ids.add(target.id)

            elif isinstance(target, ast.Tuple):
                for elt in target.elts:
                    if isinstance(elt, ast.Name):
                        self.ids.add(elt.id)

            # collect self attributes only
            elif (
                isinstance(target, ast.Attribute)
                and isinstance(target.value, ast.Name)
                and target.value.id == "self"
            ):
                self.ids.add(target.attr)

        self.generic_visit(node)

class idReplacer(ast.NodeTransformer):

    def __init__(self, ids, func_ids, class_ids):

        self.var_mapper = {}

        self._assign(ids, "var")
        self._assign(func_ids, "func")
        self._assign(class_ids, "class")

    def _assign(self, names, prefix):
        for name in sorted(names):
            if name not in self.var_mapper:
                new_name = f"{prefix}_{len(self.var_mapper)}"
                self.var_mapper[name] = new_name

    def visit_ClassDef(self, node):
        self.generic_visit(node)

        if node.name in self.var_mapper:
            node.name = self.var_mapper[node.name]

        return node

    def visit_FunctionDef(self, node):
        self.generic_visit(node)

        if node.name in self.var_mapper:
            node.name = self.var_mapper[node.name]

        used = set()
        for arg in node.args.args:
            if arg.arg in self.var_mapper:
                new_name = self.var_mapper[arg.arg]

                while new_name in used:
                    new_name += "_1"

                used.add(new_name)
                arg.arg = new_name

        return node

    def visit_Attribute(self, node):
        self.generic_visit(node)

        if (
            isinstance(node.value, ast.Name)
            and node.value.id == "self"
            and node.attr in self.var_mapper
        ):
            node.attr = self.var_mapper[node.attr]

        return node

    def visit_keyword(self, node):
        self.generic_visit(node)

        if node.arg in self.var_mapper:
            node.arg = self.var_mapper[node.arg]

        return node

    def visit_Name(self, node):
        if node.id in self.var_mapper:
            node.id = self.var_mapper[node.id]
        return node


class Id_gen:

    def get_refactored_code(self, code):
        try:
            tree = ast.parse(code)

            collector = idCollector()
            collector.visit(tree)

            replacer = idReplacer(
                collector.ids,
                collector.func_ids,
                collector.class_ids,
            )

            new_tree = replacer.visit(tree)
            ast.fix_missing_locations(new_tree)

            return ast.unparse(new_tree)

        except SyntaxError as e:
            raise ValueError(f"Syntax error in source code: {e}")


if __name__ == "__main__":

    src = """
def encrypt(pub, msg):
    ephermal_priv = ECC.generate(curve='p-521')
    ephermal_pub = ephermal_priv.public_key()
    symmetic_key = manual_shared_key(ephermal_priv, pub)
    cipher = AES.new(symmetic_key, AES.MODE_GCM)
    ciphertext, tag = cipher.encrypt_and_digest(msg)
    return cipher.nonce

class P:
    def func(self, key, pkey=None):
        self.message = "test"
        enc_method = AES.new(key, AES.MODE_CBC)
        enc_method.encrypt(self.message)
        
class P:
    def func(self, key, pkey=None):
        tst= "a"
        dc={tst:tst}
        self.message = "test"

        def func2():
            print()
        enc_key, sec_key = Kyber().encaps(pkey)
        if sec_key == key:
            print('error')
        enc_method = AES.new(sec_key, AES.MODE_CBC)
        try:
            enc_method.encrypt(self.message)
            status = enc_method.encrypt(message)
            status = enc_method.encrypt(message=message)
        except Exception as e:
            status = None 
            print(e) 
        for s in status:
            do(status[s])
        assert(status != None)
        return dc+status, enc_key 
    
p = P()
p.func(k1, k2)
p.func(k1, k2).func2()

"""

    trans = Id_gen()
    print(trans.get_refactored_code(src))
    print("*****************\n")

    src = """
def encrypt(pub, msg):
    ephermal_priv = ECC.generate(curve='p-521')
    ephermal_pub = ephermal_priv.public_key()
    symmetic_key = manual_shared_key(ephermal_priv, pub)
    cipher = AES.new(symmetic_key, AES.MODE_GCM)
    ciphertext, tag = cipher.encrypt_and_digest(msg)
    return cipher.nonce

class P:
    def func(self, key, pkey=None):
        self.message = "test"
        enc_method = AES.new(key, AES.MODE_CBC)
        enc_method.encrypt(self.message)
    def fun2(self):
        print(self.message)

"""

    trans = Id_gen()
    print(trans.get_refactored_code(src))

def func_18(var_12, var_9):
    var_5 = ECC.generate(curve='p-521')
    var_6 = var_5.public_key()
    var_15 = manual_shared_key(var_5, var_12)
    var_0 = AES.new(var_15, AES.MODE_GCM)
    (var_1, var_16) = var_0.encrypt_and_digest(var_9)
    return var_0.nonce

class class_21:

    def func_19(self, var_7, var_11=None):
        self.var_8 = 'test'
        var_4 = AES.new(var_7, AES.MODE_CBC)
        var_4.encrypt(self.var_8)

class class_21:

    def func_19(self, var_7, var_11=None):
        var_17 = 'a'
        var_2 = {var_17: var_17}
        self.var_8 = 'test'

        def func_20():
            print()
        (var_3, var_13) = Kyber().encaps(var_11)
        if var_13 == var_7:
            print('error')
        var_4 = AES.new(var_13, AES.MODE_CBC)
        try:
            var_4.encrypt(self.var_8)
            var_14 = var_4.encrypt(var_8)
            var_14 = var_4.encrypt(var_8=var_8)
        except Exception as e:
            var_14 = None
            print(e)
        for 

Current issues on feb 20

HardCodes

In [27]:
import ast


class HardcodedValues:

    SUPPORTED_TYPES = (str, int)

    def _collect_literals(self, body):
        literals = set()

        for stmt in body:
            for sub in ast.walk(stmt):

                if isinstance(
                    sub,
                    (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef),
                ):
                    continue

                if isinstance(sub, ast.Call):

                    for arg in sub.args:
                        if isinstance(arg, ast.Constant) and isinstance(
                            arg.value, self.SUPPORTED_TYPES
                        ):
                            literals.add(arg.value)

                    for kw in sub.keywords:
                        if (
                            isinstance(kw.value, ast.Constant)
                            and isinstance(
                                kw.value.value, self.SUPPORTED_TYPES
                            )
                        ):
                            literals.add(kw.value.value)

        return literals

    class Replacer(ast.NodeTransformer):
        def __init__(self, mapping):
            self.mapping = mapping

        def visit_Call(self, node):
            self.generic_visit(node)

            node.args = [
                ast.Name(id=self.mapping[a.value], ctx=ast.Load())
                if isinstance(a, ast.Constant) and a.value in self.mapping
                else a
                for a in node.args
            ]

            new_keywords = []
            for kw in node.keywords:
                if (
                    isinstance(kw.value, ast.Constant)
                    and kw.value.value in self.mapping
                ):
                    new_keywords.append(
                        ast.keyword(
                            arg=kw.arg,
                            value=ast.Name(
                                id=self.mapping[kw.value.value],
                                ctx=ast.Load(),
                            ),
                        )
                    )
                else:
                    new_keywords.append(kw)

            node.keywords = new_keywords
            return node

    def _inject_scope(self, node):

        literals = self._collect_literals(node.body)

        if not literals:
            return

        mapping = {}
        assigns = []

        for i, lit in enumerate(sorted(literals, key=str)):
            name = f"var{i}"
            mapping[lit] = name

            assign = ast.Assign(
                targets=[ast.Name(id=name, ctx=ast.Store())],
                value=ast.Constant(value=lit),
            )
            ast.fix_missing_locations(assign)
            assigns.append(assign)

        node.body = assigns + node.body

        self.Replacer(mapping).visit(node)

    def refactor(self, tree):

        for node in ast.walk(tree):
            if isinstance(node, ast.FunctionDef):
                self._inject_scope(node)

        self._inject_scope(tree)

        return ast.unparse(tree)

    def get_refactored_code(self, source_code):
        tree = ast.parse(source_code)
        return self.refactor(tree)


if __name__ == "__main__":
    src = """
print("hello",10)

def foo():
    bar("test", count=5)

def fun():
    print("hello")
    print(123)
def outer():
    print("outer")

    def inner():
        print("inner")

class A:
    def greet(self):
        print("hello", 10)
        print("hi")

"""

    t = HardcodedValues()
    print(t.get_refactored_code(src))

var0 = 10
var1 = 'hello'
print(var1, var0)

def foo():
    var0 = 5
    var1 = 'test'
    bar(var1, count=var0)

def fun():
    var0 = 123
    var1 = 'hello'
    print(var1)
    print(var0)

def outer():
    var0 = 'inner'
    var1 = 'outer'
    print(var1)

    def inner():
        print(var0)

class A:

    def greet(self):
        var0 = 10
        var1 = 'hello'
        var2 = 'hi'
        print(var1, var0)
        print(var2)


Integer ref

In [30]:
#integer refactor
import ast
import random
class ConstantRefactor(ast.NodeTransformer):
    def findFactor(self, num):
        if num%2 != 0:
            return [num/2, num/2]
        else:
            n = num//2
            return [n] + self.findFactor(n)
    def binOpFunc(self, length):
        if length <= 2:
            new_binop = ast.BinOp(
                left = ast.Constant(value=self.num_arr[length-2]),
                op=ast.Add(),
                right=ast.Constant(value=self.num_arr[length-1])
            )
            return new_binop
        else:
            new_binop = ast.BinOp(
                left = self.binOpFunc(length-1),
                op = ast.Add(),
                right = ast.Constant(value=self.num_arr[length - 1])
            )
            return new_binop
    def visit_Constant(self, node):
        r = random.randint(0, 2)
        if isinstance(node.value, int) and node.value != 0 and r:
            self.num_arr = self.findFactor(node.value)
            return self.binOpFunc(len(self.num_arr))
        return node 
    def get_refactored_code(self, code):
        try:
            tree = ast.parse(code)
            mod_tree = self.visit(tree)
            return ast.unparse(mod_tree)
        except SyntaxError as e:
            raise ValueError(f"Syntax error in source code: {e}")
       


if __name__ == "__main__":
    src = """
print("hello",10)

def foo():
    bar("test", count=5)

def fun():
    print(123)
    print("hello")
    print(123)
def outer():
    print("outer")

    def inner():
        print("inner")

class A:
    def greet(self):
        print("hello", 10)
        print("hi")

"""

    t = ConstantRefactor()
    print(t.get_refactored_code(src))  

print('hello', 5 + 2.5 + 2.5)

def foo():
    bar('test', count=2.5 + 2.5)

def fun():
    print(61.5 + 61.5)
    print('hello')
    print(61.5 + 61.5)

def outer():
    print('outer')

    def inner():
        print('inner')

class A:

    def greet(self):
        print('hello', 5 + 2.5 + 2.5)
        print('hi')


else if and relation op